# Lab 3D — Serverless Model Customization (optional)

**Persona:** Data Scientist · **Lifecycle stage:** Model build · **Run after:** Lab 3C

Labs 3B and 3C fine-tuned Llama 3.2 3B the explicit way: you chose an instance
type, handled GPU capacity, pointed a training channel at a directory, then
evaluated and registered the result as three separate steps.

This lab does the same job with **serverless model customization**, where
SageMaker owns the compute. There is no instance type to pick, no capacity
fallback to write, and training, registration, and MLflow tracking collapse
into one call.

## What you will do

1. Resolve the prompt template from the **MLflow Prompt Registry** — the same
   entry Lab 2A registered and Lab 3B trained against
2. Register Lab 2A's serverless-format dataset as a **versioned `DataSet`**
3. Fine-tune `meta-textgeneration-llama-3-2-3b-instruct` with **LoRA SFT**
4. Confirm the model **auto-registered** into its own model package group,
   still pending approval
5. Read the training curve back out of MLflow
6. Optionally **score the model with a managed evaluation job** -- a Bedrock
   model acting as judge, still with no endpoint anywhere

## This lab is optional and self-contained

It depends only on Lab 2A's output. It reads nothing from Lab 3B or 3C and
writes into **its own model package group**, so Lab 4 and Lab 5 are unaffected
whether you run it or skip it.

## What it does *not* do

It registers a model; it does not deploy one. **No endpoint is created at any
point, including by the optional evaluation in Section 10**, so there is no
hourly inference cost to forget about. Everything that needs compute --
training and scoring alike -- runs as a managed job that ends on its own.


---
## Section 1: Setup

Same environment contract as every other lab — values come from the repo-root
`.env` that `lab0-setup/setup.ipynb` wrote.


In [ ]:
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import boto3
import mlflow
from mlflow.tracking import MlflowClient

from sagemaker.ai_registry.dataset import DataSet
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.core.s3 import S3Uploader
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

# Serverless customization needs sagemaker >= 3.22.0 for sagemaker.train.
from importlib.metadata import version as _pkg_version

_sdk = _pkg_version("sagemaker")
print(f"sagemaker SDK: {_sdk}")
assert tuple(int(p) for p in _sdk.split(".")[:2]) >= (3, 22), (
    f"serverless model customization needs sagemaker>=3.22.0, found {_sdk}. "
    "Re-run lab0-setup/setup.ipynb."
)

sess = Session()
region = sess.boto_region_name
bucket = sess.default_bucket()

# Locate the repo-root .env loader.
for _candidate in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_candidate, "workshop_env.py")):
        sys.path.insert(0, _candidate)
        break


def env_from_file(key, default=""):
    """Return one key's value from the repo-root .env, or `default`.

    Reads the FILE, not os.environ, on purpose: Studio pre-seeds
    MLFLOW_TRACKING_URI with the domain's DEFAULT MLflow app, and the shared
    loader uses override=False, so an ambient value would silently win over the
    project's own. Labs 2A and 3B resolve it the same way.
    """
    try:
        from workshop_env import find_env

        env_path = find_env()
        if env_path:
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line.startswith(f"{key}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    except Exception as exc:
        print(f"  (could not read .env: {type(exc).__name__}: {exc})")
    return default


### The execution role has to be passed explicitly

This is the first place the serverless path differs from Lab 3B in a way that
will stop you. The SDK validates the role's **trust policy** before it submits
anything, and a role that SageMaker cannot assume is rejected up front:

```
RoleValidationError: IAM role '...' cannot be used for 'training' workloads.
Its trust policy does not allow 'sagemaker.amazonaws.com' to assume it
```

In Studio the ambient role is already a SageMaker execution role, so this is
mostly invisible. On a **local kernel** — where your credentials are an SSO or
IAM user role that SageMaker cannot assume — it fails immediately. Reading
`SAGEMAKER_EXEC_ROLE` from `.env` works in both places, which is why every
`DataSet` and trainer call below passes `role=` rather than relying on a
default.


In [ ]:
role = env_from_file("SAGEMAKER_EXEC_ROLE")
role_source = ".env (SAGEMAKER_EXEC_ROLE)"
if not role:
    try:
        role = get_execution_role()
        role_source = "get_execution_role()"
    except Exception as exc:
        raise RuntimeError(
            "Could not resolve a SageMaker execution role. Run "
            "lab0-setup/setup.ipynb to populate SAGEMAKER_EXEC_ROLE in .env. "
            f"get_execution_role() said: {type(exc).__name__}: {exc}"
        ) from exc

# Lab 2A writes under the user-profile prefix; resolve the same one so this lab
# reads the dataset that participant actually produced.
profile_name = env_from_file("SAGEMAKER_USER_PROFILE")
if not profile_name:
    raise RuntimeError(
        "SAGEMAKER_USER_PROFILE is not set in .env. Run lab0-setup/setup.ipynb "
        "-- Lab 2A uses it to build the S3 prefix this lab reads."
    )

profile_name = profile_name[0].upper() + profile_name[1:]

# Lab 2A writes the flat prompt/completion pair this lab needs to a prefix of
# its own, deliberately apart from Lab 3B's "dolly_dataset" training channel:
# JumpStart globs every .jsonl in a channel into one dataset, so a file with
# this lab's schema in there fails Lab 3B's training job on a schema cast.
#
# The name matters, not just the separation. S3 keys have no directories, so
# "dolly_dataset_sft" would still sit INSIDE a channel listed as
# "dolly_dataset" -- string prefix, not folder. "dolly_sft" does not.
DATASET_PREFIX = "dolly_sft"
data_location = f"s3://{bucket}/{profile_name}/{DATASET_PREFIX}"


print(f"Region        : {region}")
print(f"Bucket        : {bucket}")
print(f"Role          : {role}")
print(f"  source      : {role_source}")
print(f"Profile       : {profile_name}")
print(f"Lab 2A output : {data_location}")


### Region check

Serverless model customization is not available everywhere. There is no API to
query availability, so this is a static list that has to be maintained by hand.


In [ ]:
SERVERLESS_REGIONS = {"us-east-1", "us-west-2", "ap-northeast-1", "eu-west-1"}

if region not in SERVERLESS_REGIONS:
    raise RuntimeError(
        f"Serverless model customization is not available in {region}. "
        f"Supported: {', '.join(sorted(SERVERLESS_REGIONS))}. "
        "Lab 3B (instance-based fine-tuning) works in far more regions -- "
        "use that instead, or redeploy the workshop in a supported region."
    )
print(f"\u2713 {region} supports serverless model customization")


---
## Section 2: Why the base model changes

Lab 3B fine-tunes `meta-textgeneration-llama-3-2-3b` — the **base** model.
Serverless customization has no recipe for it. Asking for one fails:

```
ValueError: No recipes found for model 'meta-textgeneration-llama-3-2-3b'
            with customization technique: SFT
```

The same call against the **instruct** variant succeeds, for all four
techniques:

| Model ID | SFT | DPO | RLVR | RLAIF |
|---|---|---|---|---|
| `meta-textgeneration-llama-3-2-3b` | — | — | — | — |
| `meta-textgeneration-llama-3-2-3b-instruct` | yes | yes | yes | yes |

**This changes what the lab demonstrates, not just a string.** Lab 3B starts
from a model that does not reliably follow instructions, so fine-tuning teaches
it the task *and* the response format. Here the starting point already follows
instructions, so SFT is adapting an instruction-follower to your summarization
style and length conventions. Expect a lower starting loss and a smaller
relative improvement — that is the model being better to begin with, not the
training working less well.

The cell below asks the recipe catalogue directly rather than trusting the
table, because recipe availability changes over time.


In [ ]:
from sagemaker.train import list_hyperparameters

BASE_MODEL = "meta-textgeneration-llama-3-2-3b-instruct"
TECHNIQUE = "SFT"
TRAINING_TYPE = TrainingType.LORA

try:
    _hp_catalogue = list_hyperparameters(BASE_MODEL, TECHNIQUE, "LORA")
    print(f"\u2713 {BASE_MODEL}")
    print(f"  {TECHNIQUE} + LORA recipe available")
except ValueError as exc:
    raise RuntimeError(
        f"No {TECHNIQUE} recipe for {BASE_MODEL} in {region}. The serverless "
        "model catalogue changes over time -- check the SageMaker console for "
        f"a currently supported instruct model. Original error: {exc}"
    ) from exc

# Lab 3B's base model, for contrast. Expected to have no recipe.
try:
    list_hyperparameters("meta-textgeneration-llama-3-2-3b", TECHNIQUE, "LORA")
    print("\n  note: the base model now HAS an SFT recipe -- the markdown above is stale")
except ValueError as exc:
    print(f"\n  Lab 3B's base model, for contrast:")
    print(f"    meta-textgeneration-llama-3-2-3b -> {str(exc)[:90]}")


---
## Section 3: Prompt lineage

Lab 2A registered the prompt template in the MLflow Prompt Registry and Lab 3B
trained against it. This lab trains on the *same* template — but it arrives by
a different route, and the difference is worth understanding.

| | Lab 3B (JumpStart) | Lab 3D (serverless) |
|---|---|---|
| Who renders the template | the training container, at train time | Lab 2A, at data-prep time |
| How it reaches training | `template.json` in the training channel | baked into the dataset's `prompt` field |
| Needs `template.json` | yes — the container reads it off disk | no |

Because Lab 2A renders both file pairs from the one registry entry, the prompt
text is byte-identical across the two paths. This cell resolves the registry
entry so the version can be recorded as provenance on the training run — the
same lineage Lab 3C reads back.


In [ ]:
USE_CASE = "text-summarization"
PROMPT_NAME = f"{USE_CASE}-instruct"
PROMPT_ALIAS = "production"

mlflow_uri = env_from_file("MLFLOW_TRACKING_URI")
prompt_uri = ""
prompt_version_number = None
prompt_template_text = ""

if mlflow_uri:
    mlflow.set_tracking_uri(mlflow_uri)
    # Same accessor Labs 2A and 3C use. Note it must be mlflow.genai, not
    # MlflowClient: only the module-level load_prompt accepts link_to_model,
    # and MlflowClient.load_prompt raises TypeError on it.
    import mlflow.genai as genai

    try:
        _prompt = genai.load_prompt(
            f"prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}",
            allow_missing=True,   # returns None instead of raising
            link_to_model=False,  # no active run here; this is only a lookup
        )
    except Exception as exc:
        _prompt = None
        print(f"\u26a0 Prompt lookup failed: {type(exc).__name__}: {str(exc)[:140]}")

    if _prompt is not None:
        prompt_version_number = _prompt.version
        prompt_uri = f"prompts:/{PROMPT_NAME}/{_prompt.version}"
        prompt_template_text = _prompt.template
        print(f"\u2713 Resolved {PROMPT_NAME}@{PROMPT_ALIAS}")
        print(f"  version: {prompt_version_number}")
        print(f"  uri:     {prompt_uri}")
    else:
        print(f"\u26a0 No prompt registered as {PROMPT_NAME}@{PROMPT_ALIAS}.")
        print("  Training still works -- the dataset already carries the rendered")
        print("  prompt text. What is lost is the lineage record: the model package")
        print("  will be stamped UNVERIFIED instead of a prompt version.")
        print("  Re-run Lab 2A's prompt registration cell to restore it.")
else:
    print("\u26a0 No MLFLOW_TRACKING_URI in .env -- skipping prompt lineage.")
    print("  Run lab0-setup/setup.ipynb to populate it.")


---
## Section 4: Register the dataset

Serverless customization does not take an S3 path. It takes a **registered
`DataSet`**, which is a versioned artifact — and the version number is part of
the lineage, in the same way the prompt version is.

Two things to know about what this creates:

- The `DataSet` is stored as **SageMaker Hub content**, so the ARN looks like
  `.../hub-content/<hub-id>/DataSet/<name>/1.0.0`. The first call creates a
  private hub in your account; later calls reuse it.
- The required schema is flat `{"prompt", "completion"}` JSONL. Lab 2A's
  `train_sft.jsonl` / `test_sft.jsonl` are exactly that. The three files Lab 3B
  uses will **not** validate here, which is why Lab 2A emits both pairs.

Registration is fast — measured at **7.8 seconds** for 840 records.


In [ ]:
SFT_TRAIN_FILE = "train_sft.jsonl"
SFT_EVAL_FILE = "test_sft.jsonl"

sft_train_s3 = f"{data_location}/{SFT_TRAIN_FILE}"
sft_eval_s3 = f"{data_location}/{SFT_EVAL_FILE}"

# Confirm Lab 2A produced the serverless-format files before going further --
# a missing file here means Lab 2A ran before this lab existed.
from sagemaker.core.s3 import S3Downloader

_present = set(S3Downloader.list(f"{data_location}/"))
_missing = [p for p in (sft_train_s3, sft_eval_s3) if p not in _present]
if _missing:
    raise FileNotFoundError(
        f"Lab 2A has not produced {', '.join(Path(m).name for m in _missing)} at "
        f"{data_location}. Re-run lab2-data-prep/lab-2a-fine-tuning-data-prep.ipynb "
        "-- the cell titled 'Also render the serverless format' writes them, and "
        "Section 7 uploads them to this prefix. If you ran an earlier version of "
        "Lab 2A they are in the dolly_dataset prefix instead; re-running Lab 2A "
        "moves them here and clears them out of Lab 3B's training channel."
    )

print("Lab 2A objects found:")
for obj in sorted(_present):
    _tag = "  <- used by this lab" if obj in (sft_train_s3, sft_eval_s3) else ""
    print(f"  {obj}{_tag}")


In [ ]:
DATASET_NAME = f"{USE_CASE}-serverless"

# DataSet.create() bumps the version when the name already exists, so re-running
# this lab produces 1.0.0, 1.0.1, ... rather than failing or silently
# overwriting. Each training run therefore pins an exact dataset version.
_t0 = time.time()
train_dataset = DataSet.create(
    name=DATASET_NAME,
    source=sft_train_s3,
    role=role,
    sagemaker_session=sess,
    wait=True,
)
print(f"\n\u2713 Training dataset registered in {time.time() - _t0:.1f}s")
print(f"  arn: {train_dataset.arn}")

_t0 = time.time()
eval_dataset = DataSet.create(
    name=f"{DATASET_NAME}-eval",
    source=sft_eval_s3,
    role=role,
    sagemaker_session=sess,
    wait=True,
)
print(f"\u2713 Validation dataset registered in {time.time() - _t0:.1f}s")
print(f"  arn: {eval_dataset.arn}")


### Why pass a validation dataset at all

`SFTTrainer` has a `train_val_split_ratio` hyperparameter, defaulting to `0.9`,
so it will happily carve its own validation split out of the training data.

This lab passes Lab 2A's `test.jsonl` as an explicit `validation_dataset`
instead. Lab 2A's split is **stratified and leakage-checked** — it asserts no
record appears on both sides. Letting the trainer re-split would throw that
work away and quietly reintroduce the possibility of leakage. When you have a
vetted split, use it.


---
## Section 5: Model package group

The trained model registers itself, so the group has to exist first. Lab 3D
uses **its own group**, separate from the timestamped group Lab 3C creates.

That separation is deliberate. If a serverless LoRA model and a JumpStart model
landed in the same group, "deploy the approved version" would become ambiguous,
and Lab 4 would stop being deterministic.


In [ ]:
MODEL_PACKAGE_GROUP_NAME = f"{USE_CASE}-serverless"

from botocore.exceptions import ClientError

try:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description=(
            "Lab 3D - serverless LoRA SFT of Llama 3.2 3B Instruct for summarization"
        ),
    )
    print(f"\u2713 Created model package group: {MODEL_PACKAGE_GROUP_NAME}")
except ClientError as exc:
    if exc.response["Error"]["Code"] in ("ResourceInUse", "ValidationException"):
        model_package_group = ModelPackageGroup.get(
            model_package_group_name=MODEL_PACKAGE_GROUP_NAME
        )
        print(f"\u2713 Reusing existing model package group: {MODEL_PACKAGE_GROUP_NAME}")
    else:
        raise

print(f"  arn: {model_package_group.model_package_group_arn}")


---
## Section 6: Configure the trainer

Notice what is **absent** compared with Lab 3B: no `Compute`, no instance type,
no capacity-aware fallback list. SageMaker owns the hardware, so none of those
are decisions you get to make — or have to handle when capacity is short.


In [ ]:
EXPERIMENT_NAME = f"{USE_CASE}-serverless"

trainer_kwargs = dict(
    model=BASE_MODEL,
    training_type=TRAINING_TYPE,
    model_package_group=model_package_group,
    training_dataset=train_dataset.arn,
    validation_dataset=eval_dataset.arn,
    s3_output_path=f"s3://{bucket}/lab3d-serverless/output/",
    sagemaker_session=sess,
    role=role,
)

# MLflow auto-wires even with no arguments -- it creates an experiment named
# after the model package group. Naming it explicitly keeps this lab's runs
# findable next to Lab 3A/3B's, instead of in a separate auto-named experiment.
if mlflow_uri:
    trainer_kwargs["mlflow_resource_arn"] = mlflow_uri
    trainer_kwargs["mlflow_experiment_name"] = EXPERIMENT_NAME
    print(f"MLflow experiment: {EXPERIMENT_NAME}")
else:
    print("MLflow not configured -- training metrics will not be tracked")


### License acceptance

Llama 3.2 3B Instruct is a **gated model**. The SDK enforces this at trainer
construction, not at submission:

```
ValueError: Model 'meta-textgeneration-llama-3-2-3b-instruct' is a gated model
            and requires EULA acceptance. Please set accept_eula=True
```

This model is licensed under the **Llama 3.2 Community License Agreement**.
Review the terms before continuing:
<https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/blob/main/LICENSE.txt>

**You must read and agree to the license to proceed.** Set `ACCEPT_EULA = True`
in the cell below only if you accept it. Lab 3B asks you the same question for
the base model.


In [ ]:
# Set to True only if you have read and accept the Llama 3.2 Community License
# linked above. Left False, this cell stops the lab here by design.
ACCEPT_EULA = True

if not ACCEPT_EULA:
    raise SystemExit(
        "ACCEPT_EULA is False. Review the Llama 3.2 Community License linked "
        "above and set ACCEPT_EULA = True in this cell to continue."
    )

trainer = SFTTrainer(accept_eula=ACCEPT_EULA, **trainer_kwargs)
print("\u2713 Trainer configured\n")

hp = trainer.hyperparameters
print("Recipe defaults:")
print(f"  global_batch_size     : {hp.global_batch_size}")
print(f"  learning_rate         : {hp.learning_rate}")
print(f"  max_epochs            : {hp.max_epochs}")
print(f"  lora_rank             : {hp.lora_rank}")
print(f"  lora_alpha            : {hp.lora_alpha}")
print(f"  dataset_max_len       : {hp.dataset_max_len}")
print("\nFor every parameter, its range and valid options: hp.get_info()")


### Optional: override hyperparameters

The defaults are what the measured 11.5-minute benchmark below used. Each
override is validated against the recipe, so an out-of-range value fails
immediately with the allowed range rather than wasting a training run.

`max_epochs` defaults to **1**, which is the single biggest lever on both
quality and duration here.


In [ ]:
# Uncomment and adjust. hp.get_info() prints the valid range for each one.

# trainer.hyperparameters.max_epochs = 3            # 1 - 100
# trainer.hyperparameters.learning_rate = 0.0002    # 1e-07 - 0.001
# trainer.hyperparameters.global_batch_size = 128   # 64, 128, 256, 512, 1024
# trainer.hyperparameters.lora_rank = 32            # 8, 16, 32, 64, 128

print(f"max_epochs        = {trainer.hyperparameters.max_epochs}")
print(f"learning_rate     = {trainer.hyperparameters.learning_rate}")
print(f"global_batch_size = {trainer.hyperparameters.global_batch_size}")
print(f"lora_rank         = {trainer.hyperparameters.lora_rank}")


---
## Section 7: Train

Measured on Lab 2A's 840 records with the defaults above:

| Phase | Duration |
|---|---|
| Waiting for capacity | 94s |
| Preparing instances | 31s |
| Downloading input data | 62s |
| Downloading training image | 219s |
| **Training** | **187s** |
| Uploading model | 62s |
| **Total** | **688s (11.5 min)** |

Most of that is not training. Actual gradient work was about three minutes; the
rest is capacity acquisition and a large container image. A bigger dataset
mainly grows the training phase, so the fixed overhead matters less as you
scale up.

The cell submits with `wait=False` and polls, so you get phase-by-phase
progress instead of a blocking call with nothing to look at.


In [ ]:
_t0 = time.time()
training_job = trainer.train(wait=False)
print(f"Submitted in {time.time() - _t0:.1f}s")
print(f"  job name: {training_job.training_job_name}")
print(f"  status  : {training_job.training_job_status}\n")

sm_client = boto3.client("sagemaker", region_name=region)
_seen = set()
_start = time.time()

while True:
    _desc = sm_client.describe_training_job(
        TrainingJobName=training_job.training_job_name
    )
    for _transition in _desc.get("SecondaryStatusTransitions", []):
        _key = (_transition["Status"], _transition.get("StatusMessage", ""))
        if _key not in _seen:
            _seen.add(_key)
            print(f"[{time.time() - _start:6.0f}s] {_transition['Status']:<12} "
                  f"{_transition.get('StatusMessage', '')[:70]}")

    _status = _desc["TrainingJobStatus"]
    if _status != "InProgress":
        print(f"\n{_status} after {time.time() - _start:.0f}s")
        if _desc.get("FailureReason"):
            print(f"FailureReason: {_desc['FailureReason'][:500]}")
        break

    time.sleep(30)

assert _status == "Completed", f"training ended as {_status}, not Completed"
model_artifacts = _desc.get("ModelArtifacts", {}).get("S3ModelArtifacts", "")
print(f"\nModel artifacts: {model_artifacts}")


---
## Section 8: The model registered itself

In Lab 3C you evaluated the model and called `register()` yourself. Here it
already happened — completing the training job created the model package.

Two details matter for governance:

- **`ModelApprovalStatus` is `None`.** Auto-registration does not mean
  auto-approval. The Lab 4 promotion gate still applies, so nothing reaches an
  endpoint without someone approving it.
- **The LoRA adapter is merged into the final model**
  (`merge_lora_on_final_save: True`), so the artifact is a complete model, not
  an adapter you have to compose at load time.


In [ ]:
_packages = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)["ModelPackageSummaryList"]

print(f"Model packages in {MODEL_PACKAGE_GROUP_NAME}:\n")
for _pkg in _packages:
    print(f"  v{_pkg['ModelPackageVersion']:<3} {_pkg['ModelPackageStatus']:<12} "
          f"approval={_pkg.get('ModelApprovalStatus', 'None'):<10} "
          f"{_pkg['CreationTime']:%Y-%m-%d %H:%M}")

latest_model_package_arn = _packages[0]["ModelPackageArn"] if _packages else ""
print(f"\nLatest: {latest_model_package_arn}")
print("\nApproval is still pending -- Lab 4's gate is unchanged by auto-registration.")


### Record the prompt lineage on the model package

The dataset already contains the rendered prompt, so the *model* is tied to the
prompt implicitly. That is not good enough for governance: six months from now
nobody can tell which registry version produced it by looking at the package.

Writing it into `CustomerMetadataProperties` makes the link explicit and
queryable — the same provenance Lab 3C records.

> Metadata values are validated against
> `[\p{L}\p{Z}\p{N}_.:/=+\-@]*`, which rejects characters as ordinary as
> parentheses. `metadata_safe()` strips anything outside that set, matching
> Lab 3C's sanitizer.


In [ ]:
_DISALLOWED = re.compile(r"[^\w\s.:/=+\-@]", re.UNICODE)


def metadata_safe(value):
    """Strip characters CustomerMetadataProperties rejects."""
    return _DISALLOWED.sub("", str(value)).strip()


if latest_model_package_arn:
    _metadata = {
        "use_case": USE_CASE,
        "customization": "serverless-lora-sft",
        "base_model": BASE_MODEL,
        "training_dataset_arn": train_dataset.arn,
        "validation_dataset_arn": eval_dataset.arn,
        "training_job_name": training_job.training_job_name,
        "prompt_name": PROMPT_NAME,
        "prompt_version": str(prompt_version_number) if prompt_version_number else "UNVERSIONED",
        "prompt_version_uri": prompt_uri or "UNVERIFIED-no-registry-entry",
        "produced_by_lab": "lab-3d-serverless-fine-tuning",
    }
    _metadata = {k: metadata_safe(v) for k, v in _metadata.items()}

    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        CustomerMetadataProperties=_metadata,
    )
    print("\u2713 Provenance recorded on the model package\n")
    for _k, _v in _metadata.items():
        print(f"  {_k:24} {_v[:70]}")
else:
    print("\u26a0 No model package found -- skipping provenance.")


---
## Section 9: Read the training curve from MLflow

MLflow tracking is automatic. Even with no MLflow arguments at all, the trainer
creates an experiment named after the model package group and logs to it.

Logged metrics go well beyond loss: sequence-length distribution for the
dataset (`data/train_seq_len_*`) and GPU peak memory (`perf/gpu_peak_memory_*`),
which is the useful signal when deciding whether a larger `global_batch_size`
would fit.

The trainer underneath is Verl, so the loss curve is **`train/loss`**. Lab 3B's
JumpStart path logs different names — a reason to list what was actually logged
rather than hardcoding a metric name.


In [ ]:
if not mlflow_uri:
    print("MLflow not configured -- nothing to read.")
else:
    mlflow.set_tracking_uri(mlflow_uri)
    _client = MlflowClient()

    _run = None
    for _exp in _client.search_experiments():
        for _candidate in _client.search_runs([_exp.experiment_id], max_results=100):
            if _candidate.data.tags.get("mlflow.runName") == training_job.training_job_name:
                _run = _candidate
                break
        if _run:
            print(f"Experiment: {_exp.name}")
            break

    if _run is None:
        print(f"No MLflow run named {training_job.training_job_name} found yet.")
        print("Metrics can lag job completion by a few seconds -- re-run this cell.")
    else:
        print(f"Run id    : {_run.info.run_id}\n")
        _loss = [m for m in _run.data.metrics if "loss" in m.lower()]
        print(f"Loss metrics logged: {sorted(_loss)}\n")

        for _metric in ("train/loss", "train/per_token_loss"):
            if _metric in _run.data.metrics:
                _history = _client.get_metric_history(_run.info.run_id, _metric)
                print(f"  {_metric}: {len(_history)} steps, "
                      f"{_history[0].value:.4f} -> {_history[-1].value:.4f}")

        for _metric in sorted(_run.data.metrics):
            if _metric.startswith("perf/"):
                print(f"  {_metric}: {_run.data.metrics[_metric]:.2f}")


In [ ]:
# Plot whichever loss curve was logged, rather than assuming a metric name.
if mlflow_uri and _run is not None:
    import matplotlib.pyplot as plt

    _available = set(_run.data.metrics)
    _candidates = [("train/loss", "total_loss"), ("val/loss", "val_eval_total_loss")]
    _to_plot = [next(n for n in names if n in _available)
                for names in _candidates if _available & set(names)]

    if not _to_plot:
        print(f"No expected loss curves. Logged: {sorted(_available)}")
    else:
        _fig, _axes = plt.subplots(1, len(_to_plot),
                                   figsize=(6 * len(_to_plot), 3.5), squeeze=False)
        for _ax, _metric in zip(_axes[0], _to_plot):
            _history = _client.get_metric_history(_run.info.run_id, _metric)
            _ax.plot([h.step for h in _history], [h.value for h in _history],
                     linewidth=2, marker="o", markersize=4)
            _ax.set_xlabel("Step")
            _ax.set_ylabel("Loss")
            _ax.set_title(_metric, fontweight="bold")
            _ax.grid(True, alpha=0.3)
        plt.suptitle(f"Lab 3D - {training_job.training_job_name}", fontweight="bold")
        plt.tight_layout()
        plt.show()


---
## Section 10: Evaluate the model (optional)

Section 8 registered a model package with **no quality metrics attached**. A
reviewer looking at it has the provenance but no evidence, so there is nothing
to approve on. This section supplies the evidence -- and it does it the same
way the rest of the lab does the training: as a **managed job, with no endpoint
to provision, size, or tear down**.

`sagemaker.train.evaluate.LLMAsJudgeEvaluator` runs scoring as a SageMaker
pipeline that invokes a **Bedrock foundation model as the judge**. You hand it
the model, the registered `DataSet`, and a metric list; it returns an execution
you can poll.

### Why not ROUGE and BLEU, like Lab 3C

Because an endpoint would be the only way to get them here, and standing one up
would contradict the lab. Three consequences worth being explicit about:

| | Lab 3C | Lab 3D |
|---|---|---|
| Metric family | ROUGE / BLEU, n-gram overlap against a reference | judge scores: correctness, completeness, faithfulness |
| Compute you manage | a GPU endpoint per model | none |
| Needs a reference summary | yes | yes -- the `completion` field |
| Comparable across the two labs | -- | **no, by design** |

The last row is a feature, not a gap. Lab 3B fine-tunes the **base** checkpoint
and this lab the **instruct** one, so a 3C-vs-3D metric comparison was always
comparing two different starting points. Dropping it removes a misleading
number. If you want a like-for-like comparison, compare within a lab.

### What this costs, measured

No endpoint, so nothing bills hourly. You pay for the pipeline's compute and
the judge's tokens. A full run over Lab 2A's 334-record eval split, scoring
three metrics with Nova Micro as judge:

| Pipeline step | Duration |
|---|---|
| `CreateEvaluationAction` | 3s |
| `EvaluateCustomInferenceModel` (generate 334 summaries) | 11 min |
| `EvaluateCustomModelMetrics` (judge scores them) | ~32 min |
| `AssociateLineage` | seconds |
| **Total** | **~44 min** |

Nothing was provisioned in the account beyond the pipeline itself -- verified by
listing endpoints during the run. Still opt-in, because it is not free.

In [ ]:
# ---------------------------------------------------------------------------
# Gates. Evaluation is opt-in; nothing below submits until the first flag is on.
# ---------------------------------------------------------------------------
RUN_JUDGE_EVALUATION = True   # submits ONE managed evaluation job (no endpoint)
EVALUATE_BASE_MODEL = True    # also score the instruct base, for a delta

# Bedrock LLM-as-judge built-in metrics. The "Builtin." prefix is optional --
# the SDK strips it. Correctness and completeness are scored against the
# reference `completion`; faithfulness catches summaries that invent facts,
# which is the failure mode Lab 2A's groundedness gate screens for in the
# training data.
JUDGE_METRICS = ["Correctness", "Completeness", "Faithfulness"]

# Same prefix family as the training job's output in Section 7.
EVAL_OUTPUT_S3 = f"s3://{bucket}/lab3d-serverless/eval/"

print(f"judge metrics   : {JUDGE_METRICS}")
print(f"eval output     : {EVAL_OUTPUT_S3}")
print(f"run evaluation  : {RUN_JUDGE_EVALUATION}")
print(f"score base too  : {EVALUATE_BASE_MODEL}")

### Choosing a judge, without hardcoding one

The SDK reads its allowlist of judge-capable models from a service-maintained
file in the JumpStart cache bucket, and that list is **a superset**: it is
described in the SDK source as possibly including models past end of life.
Picking one off the list is how you get this, minutes in:

```
ResourceNotFoundException: This model version has reached the end of its life.
Use an active model version instead.
```

So the cell below intersects two sources of truth -- the SDK's supported-judge
list and Bedrock's own `ACTIVE` lifecycle status -- and prefers a model that
supports `ON_DEMAND` invocation. Models that are `INFERENCE_PROFILE`-only need
a cross-region profile id (`us.` prefix) rather than the bare model id, which is
a second way to get a late failure.

This is the same discipline as resolving the serving image instead of pinning a
tag: the catalogue moves, so ask it rather than assume it.

In [ ]:
# Resolve a judge model that is BOTH judge-capable and currently in service.
#
# Two lookups, and BOTH are optional: an environment provisioned before the
# Lab 3D IAM statements existed will be denied them, e.g.
#   AccessDeniedException ... not authorized to perform:
#   bedrock:ListFoundationModels
# A workshop lab should not dead-end on a permission the participant cannot
# grant themselves, so each lookup degrades to a default that is known to work
# rather than raising.
_SUPPORTED_KEY = "fmhMetadata/supported-llmaj-judge-models.json"

# Measured working default: submitted as the bare id, the service resolved it
# to the cross-region profile "us.amazon.nova-micro-v1:0" itself.
_DEFAULT_JUDGE = "amazon.nova-micro-v1:0"

_supported, _active, _why = set(), {}, []

try:
    _doc = json.loads(boto3.client("s3", region_name=region).get_object(
        Bucket=f"jumpstart-cache-prod-{region}", Key=_SUPPORTED_KEY)["Body"].read())
    _supported = {m["model_id"] for m in _doc["supported_judge_models"]}
    print(f"supported-judge list: {len(_supported)} models "
          f"(updated {_doc.get('last_updated', 'unknown')})")
except Exception as exc:
    _why.append(f"judge list unreadable ({type(exc).__name__})")

try:
    _active = {m["modelId"]: m.get("inferenceTypesSupported", [])
               for m in boto3.client("bedrock", region_name=region)
               .list_foundation_models()["modelSummaries"]
               if m.get("modelLifecycle", {}).get("status") == "ACTIVE"}
    print(f"ACTIVE Bedrock models in {region}: {len(_active)}")
except Exception as exc:
    _why.append(f"model catalogue unreadable ({type(exc).__name__})")

if _supported and _active:
    _usable = sorted(_supported & _active.keys())
    if not _usable:
        raise RuntimeError(
            f"No judge-capable Bedrock model is ACTIVE in {region}. Check "
            "Bedrock model access for this account, or run this lab in a "
            "region with broader Bedrock availability."
        )
    _on_demand = [m for m in _usable if "ON_DEMAND" in _active[m]]
    JUDGE_MODEL = (_on_demand or _usable)[0]
    print(f"\njudge-capable and ACTIVE ({len(_usable)}):")
    for _m in _usable:
        print(f"  {_m:48} {','.join(_active[_m]):28}"
              + ("  <- selected" if _m == JUDGE_MODEL else ""))
    _eol = sorted(_supported - _active.keys())
    print(f"\nlisted as supported but NOT active here ({len(_eol)}) -- "
          "picking one of these is how a run fails minutes in")
else:
    JUDGE_MODEL = _DEFAULT_JUDGE
    print("\n\u26a0 Falling back to the default judge: " + JUDGE_MODEL)
    for _r in _why:
        print(f"   {_r}")
    print("   Both lookups are read-only and are granted by the workshop's IAM")
    print("   stack (BedrockFoundationModelCatalogueRead and")
    print("   JumpStartCacheReadForJudgeList in templates/2-iam.yaml). Without")
    print("   them this lab still runs -- it just cannot warn you if the")
    print("   default judge is ever retired.")

# Note: pass the BARE model id. The service resolves a cross-region inference
# profile itself.
print(f"\njudge model: {JUDGE_MODEL}")

### What this needs that the rest of the lab does not

Scoring runs as a **Bedrock** evaluation job, so two things must be true beyond
what training needed:

1. **The execution role can create and read Bedrock evaluation jobs.** The SDK
   validates this before submitting and fails fast:

   ```
   RoleValidationError: IAM role '...' cannot be used for 'model_eval'
   workloads. Missing permissions: bedrock:CreateEvaluationJob,
   bedrock:GetEvaluationJob
   ```

   The workshop's IAM stack grants these (`BedrockModelEvaluationLab3D` in
   `templates/2-iam.yaml`). A domain deployed before that statement existed
   needs `ONLY_STACKS=iam ./scripts/deploy-workshop.sh` -- minutes, not a full
   redeploy.

2. **Bedrock model access is enabled** for the judge in this account and
   region.

The cell below checks both the cheap way: it builds the evaluator and calls
`evaluate(dry_run=True)`, which runs the SDK's validation without submitting
anything.

In [ ]:
import sagemaker.train.evaluate.execution as _eval_exec
from sagemaker.train.evaluate import LLMAsJudgeEvaluator

# ---------------------------------------------------------------------------
# Workaround for an SDK bug, measured on sagemaker 3.28.0 / sagemaker-train
# 1.28.0 (the latest releases at the time of writing).
#
# execution.py builds the pipeline's tags as sagemaker.core.shapes.Tag objects
# (lines 77 and 91, under a comment asserting that Tag objects are what
# Pipeline.create needs). They are not: that Pipeline.create validates `tags`
# against a different Tag model and rejects both the shapes.Tag instance and a
# {"Key","Value"} dict, so EVERY evaluator in this family fails before
# submitting anything:
#
#   Unexpected error: 1 validation error for Pipeline.create
#   tags.0  Input should be a valid dictionary or instance of Tag
#
# What it does accept is a {"key","value"} dict. Replacing the module's Tag
# symbol with a factory for those is the smallest fix that does not touch the
# installed package. Delete this block once the SDK is fixed -- the probe below
# makes it inert as soon as Tag already yields a dict.
_probe = _eval_exec.Tag(key="k", value="v")
if not isinstance(_probe, dict):
    _eval_exec.Tag = lambda key, value: {"key": key, "value": value}
    print("applied SDK tag workaround (sagemaker-train pipeline tags)")
# ---------------------------------------------------------------------------

# The model under test. A model package ARN is accepted directly -- including a
# pending-approval one, which matters here: Section 8's package is deliberately
# NOT approved, and the old endpoint-based version of this section had to work
# around CreateModel refusing a non-Approved package.
_model_under_test = latest_model_package_arn or training_job.training_job_name

judge_evaluator = LLMAsJudgeEvaluator(
    model=_model_under_test,
    dataset=eval_dataset,              # the versioned DataSet from Section 4
    evaluator_model=JUDGE_MODEL,
    builtin_metrics=JUDGE_METRICS,
    s3_output_path=EVAL_OUTPUT_S3,
    mlflow_resource_arn=mlflow_uri or None,
    mlflow_experiment_name=MODEL_PACKAGE_GROUP_NAME,
    mlflow_run_name=f"judge-eval-{training_job.training_job_name[-14:]}",
    model_package_group=MODEL_PACKAGE_GROUP_NAME,
    evaluate_base_model=EVALUATE_BASE_MODEL,
    role=role,
    region=region,
)
print("\u2713 evaluator constructed")
print(f"  model     : {_model_under_test.split('/')[-2:]}")
print(f"  dataset   : {eval_dataset.arn.split('/')[-2]} v{eval_dataset.arn.split('/')[-1]}")
print(f"  judge     : {JUDGE_MODEL}")

# Validation only -- submits nothing.
try:
    judge_evaluator.evaluate(dry_run=True)
    print("\n\u2713 dry run passed: role, dataset and judge all accepted")
    PREFLIGHT_OK = True
except Exception as exc:
    PREFLIGHT_OK = False
    _msg = str(exc)
    print(f"\n\u2717 dry run failed: {type(exc).__name__}")
    print("  " + _msg[:700].replace("\n", "\n  "))
    if "bedrock:CreateEvaluationJob" in _msg or "model_eval" in _msg:
        print("\n  This is the IAM prerequisite above. Fix it with:")
        print("    ONLY_STACKS=iam ./scripts/deploy-workshop.sh")
        print("  then restart the kernel and re-run this cell.")

In [ ]:
import contextlib

# ---------------------------------------------------------------------------
# Submit. No instance type, no capacity fallback, no endpoint to clean up --
# the same contract as the training job in Section 7.
# ---------------------------------------------------------------------------
eval_execution = None

if not RUN_JUDGE_EVALUATION:
    print("RUN_JUDGE_EVALUATION is False -- skipping evaluation.")
    print("The model stays registered with provenance but no quality evidence,")
    print("which is a reviewable state only if someone else supplies the")
    print("evidence. Set the flag to True to produce it here.")
elif not PREFLIGHT_OK:
    raise RuntimeError(
        "The dry run above did not pass, so there is no point submitting. "
        "Resolve the reported problem first."
    )
else:
    eval_execution = judge_evaluator.evaluate()
    _arn = getattr(eval_execution, "arn", None) or getattr(
        eval_execution, "pipeline_execution_arn", "")
    print(f"submitted: {_arn or eval_execution}")
    print("waiting for the evaluation pipeline (no endpoint is created)...")
    try:
        eval_execution.wait(target_status="Succeeded", poll=30, timeout=5400)
        print("\n\u2713 evaluation succeeded")
    except Exception as exc:
        print(f"\n\u26a0 wait ended with {type(exc).__name__}: {str(exc)[:300]}")
    finally:
        with contextlib.suppress(Exception):
            eval_execution.refresh()
            _st = eval_execution.status
            print(f"  overall status: {getattr(_st, 'overall_status', _st)}")
            for _step in getattr(_st, "steps", None) or []:
                print(f"    {getattr(_step, 'name', _step)}: "
                      f"{getattr(_step, 'status', '')}")
            _reason = getattr(_st, "failure_reason", "")
            if _reason:
                print(f"  failure reason: {str(_reason)[:400]}")

### Reading the scores

The evaluator logs to the MLflow experiment and run you named, so the scores
arrive the same way the training curve did in Section 9 -- no parsing of job
output required. The raw judge report also lands under `EVAL_OUTPUT_S3` if you
want the per-record detail.

This cell lists what was actually logged rather than assuming metric names:
Bedrock's judge metrics are logged under its own naming, and asserting a name
you did not verify is how a notebook reports "0.0" for a metric that was never
written.

In [ ]:
eval_metrics = {}

# Two sources, most authoritative first.
#   1. The judge job's own results file, which carries per-metric pass/fail
#      counts as well as the score -- useful evidence, not just a number.
#   2. The MLflow run, which is the lineage record.
# The MLflow run is NOT named after mlflow_run_name: the evaluator names it
# after the pipeline STEP (measured: "pipelines-<exec-id>-EvaluateCustomModelM-...").
# So it is located by the execution id, not by a name we chose.
if eval_execution is not None:
    from sagemaker.core.s3 import S3Downloader

    _exec_arn = getattr(eval_execution, "arn", "") or ""
    _exec_id = _exec_arn.rsplit("/", 1)[-1]

    # --- 1. the judge results file --------------------------------------
    _results = [o for o in S3Downloader.list(EVAL_OUTPUT_S3)
                if o.endswith("bedrock_llm_judge_results.json")
                and (not _exec_id or _exec_id in o)]
    if _results:
        _doc = json.loads(S3Downloader.read_file(sorted(_results)[-1]))
        print(f"judge model : {_doc.get('judge_model')}")
        print(f"job status  : {_doc.get('status')}")
        print(f"{'metric':28} {'score':>8} {'passed':>8} {'of':>6}")
        for _name, _r in (_doc.get("results") or {}).items():
            eval_metrics[_name] = _r["score"]
            print(f"  {_name:26} {_r['score']:8.4f} {_r['passed']:8} "
                  f"{_r['total_evaluations']:6}")
    else:
        print(f"No judge results file under {EVAL_OUTPUT_S3} yet.")

    # --- 2. the MLflow record -------------------------------------------
    if mlflow_uri:
        import mlflow
        from mlflow.tracking import MlflowClient

        mlflow.set_tracking_uri(mlflow_uri)
        _client = MlflowClient()
        _exp = _client.get_experiment_by_name(MODEL_PACKAGE_GROUP_NAME)
        _runs = _client.search_runs([_exp.experiment_id],
                                    order_by=["attributes.start_time DESC"],
                                    max_results=20) if _exp else []
        _match = [r for r in _runs
                  if (_exec_id and _exec_id in (r.info.run_name or ""))
                  or any(k.startswith("Builtin/") for k in r.data.metrics)]
        if _match:
            _run = _match[0]
            print(f"\nMLflow run: {_run.info.run_name}")
            for _k in sorted(_run.data.metrics):
                if not _k.startswith("system/"):
                    print(f"  {_k:30} {_run.data.metrics[_k]:.4f}")
        else:
            print("\nNo matching MLflow run found yet -- the step logs on exit,")
            print("so give it a moment after the pipeline reports Succeeded.")

### Verdict

Computed from whatever was logged, never asserted. With
`EVALUATE_BASE_MODEL = True` the run also scores the instruct checkpoint this
model was fine-tuned from, which is the only honest baseline available here --
and unlike the old cross-lab comparison against Lab 3C, both sides share a
starting point.

In [ ]:
if not eval_metrics:
    print("No metrics to summarize.")
else:
    # Pair up base/tuned metrics when both were scored. Names are whatever the
    # evaluator logged, so group by suffix rather than hardcoding keys.
    _base = {k: v for k, v in eval_metrics.items() if "base" in k.lower()}
    _tuned = {k: v for k, v in eval_metrics.items() if k not in _base}

    print(f"{'metric':44} {'score':>8}")
    for _k in sorted(_tuned):
        print(f"  {_k:42} {_tuned[_k]:8.4f}")

    if _base:
        print(f"\n{'baseline (instruct base model)':44}")
        for _k in sorted(_base):
            print(f"  {_k:42} {_base[_k]:8.4f}")
        print("\nRead the pairs side by side before approving: a fine-tune that")
        print("does not beat its own starting point is evidence against itself.")
    else:
        print("\nNo baseline in this run (EVALUATE_BASE_MODEL was False), so")
        print("these numbers describe the tuned model in isolation. That is")
        print("enough to spot an obviously broken fine-tune, and not enough to")
        print("claim an improvement.")

---
## Section 11: Record the evidence and decide

`ModelMetrics` is create-only, so the scores go into
`CustomerMetadataProperties` alongside the provenance from Section 8. Not as
tidy as the native field, but it is queryable and it travels with the package.

Approval stays **manual and explicit**. `APPROVE_MODEL` defaults to `False`
because a notebook that approves its own model is not a governance gate.


In [ ]:
APPROVE_MODEL = False   # set True only after reading the verdict above

if latest_model_package_arn and eval_metrics:
    _evidence = {
        "eval_method": "serverless-llm-as-judge",
        "eval_judge_model": JUDGE_MODEL,
        "eval_dataset_arn": eval_dataset.arn,
        "eval_metrics_requested": ",".join(JUDGE_METRICS),
    }
    for _k, _v in eval_metrics.items():
        _evidence[f"eval_{_k}"] = f"{_v:.4f}"

    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        CustomerMetadataProperties={k: metadata_safe(v) for k, v in _evidence.items()})
    print("\u2713 evaluation evidence recorded on the model package")
    for _k, _v in sorted(_evidence.items()):
        print(f"  {_k:24} {_v}")
elif latest_model_package_arn:
    print("No metrics to record (evaluation was skipped).")

if APPROVE_MODEL and latest_model_package_arn:
    sm_client.update_model_package(
        ModelPackageArn=latest_model_package_arn,
        ModelApprovalStatus="Approved",
        ApprovalDescription=(
            f"Approved on serverless judge evidence ({JUDGE_MODEL}): "
            + ", ".join(f"{k}={v:.4f}" for k, v in eval_metrics.items())
        )[:1024])
    print("\n\u2713 model package APPROVED -- it can now be deployed")
    print("  Note: CreateModel from a package only works once it is Approved,")
    print("  which is why evaluation above served the artifact directly.")
else:
    _status = "unknown"
    if latest_model_package_arn:
        _status = sm_client.describe_model_package(
            ModelPackageName=latest_model_package_arn).get("ModelApprovalStatus", "None")
    print(f"\nApproval status: {_status}")
    print("  Set APPROVE_MODEL = True to approve, or approve in the SageMaker")
    print("  console / via Lab 4's pipeline.")


---
## Section 12: What this lab traded away

Side by side with Labs 3B and 3C:

| | Lab 3B + 3C | Lab 3D |
|---|---|---|
| Compute | you pick the instance type | SageMaker owns it |
| GPU capacity | you write the fallback logic | not your problem |
| Techniques | whatever the container supports | SFT, DPO, RLVR, RLAIF recipes only |
| Base models | the whole JumpStart catalogue | models with a serverless recipe |
| Prompt delivery | `template.json` in the channel | rendered into the dataset |
| Dataset | S3 path | versioned `DataSet` artifact |
| Registration | explicit, in Lab 3C | automatic |
| Metrics on the package | `ModelMetrics` at creation | metadata only (create-only field) |
| Evaluation | you deploy an endpoint per model and score ROUGE/BLEU yourself | managed judge job, no endpoint |
| Evaluation metrics | n-gram overlap against a reference | judge scores from a Bedrock model |
| MLflow | you wire it up | automatic |
| Custom container | supported | not supported |

**The trade is control for operational surface.** Lab 3D removes real work —
instance sizing, capacity handling, registration plumbing — and in exchange you
can only train what a recipe already covers. For a standard LoRA SFT on a
supported model, that is a good deal. For a custom loss function, a bespoke
container, or an unsupported base model, Lab 3B is the only path.

**On cost:** `describe-training-job` returns `BillableTimeInSeconds: 0` and
`ResourceConfig: None` for serverless jobs. There are no instance-seconds to
report, so per-run cost cannot be read from the API — check Cost Explorer after
charges settle, or the serverless customization pricing page.

### What is left behind

No endpoint is created, so nothing is billing hourly. These do persist:

- the registered model package (pending approval)
- the `DataSet` versions, held as private SageMaker Hub content
- model artifacts under `s3://{bucket}/lab3d-serverless/output/`

All cheap to keep. The teardown cell below is commented out on purpose —
deleting a registered model is a governance action, not cleanup.


In [ ]:
# Teardown. Commented out deliberately: deleting a registered model package is
# a governance decision, so make it explicitly rather than by running a cell.

# for _pkg in sm_client.list_model_packages(
#         ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)["ModelPackageSummaryList"]:
#     sm_client.delete_model_package(ModelPackageName=_pkg["ModelPackageArn"])
# sm_client.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)
# DataSet.delete_by_name(name=DATASET_NAME, sagemaker_session=sess)
# DataSet.delete_by_name(name=f"{DATASET_NAME}-eval", sagemaker_session=sess)

print("Lab 3D complete.")
print(f"  model package group : {MODEL_PACKAGE_GROUP_NAME}")
print(f"  training job        : {training_job.training_job_name}")
print(f"  dataset             : {train_dataset.arn.split('/')[-2]} v{train_dataset.arn.split('/')[-1]}")
print(f"  prompt              : {prompt_uri or 'not resolved'}")
